# Auditing a care-management risk score

## Section 3 — Audit recorded active chronic conditions

**Question:** Among patients with similar commercial risk scores, do
Black and White patients have similar mean numbers of recorded active chronic conditions?


## START/RESTART HERE

Before Task 1, run every setup code cell below in order. Each setup cell is
labeled `# RUN THIS CELL FIRST`; continue until you reach **Task 1**.

The setup cells import packages and load the data.


In [ ]:
# RUN THIS CELL FIRST
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
RACE_STYLES = {
    "black": {"color": "#6A3D9A", "linestyle": "--", "marker": "o"},
    "white": {"color": "#E69F00", "linestyle": "-", "marker": "s"},
}


In [ ]:
# RUN THIS CELL FIRST
DATA_COMMIT = "daceb25bba00e65d7b05882f049e229a8bedb60c"
DATA_URL = (
    "https://gitlab.com/labsysmed/dissecting-bias/-/raw/"
    f"{DATA_COMMIT}/data/data_new.csv"
)
LOCAL_DATA_PATH = Path("data_new.csv")


def load_data(required_columns):
    """Load the course data from a local copy or the pinned public URL."""
    source = LOCAL_DATA_PATH if LOCAL_DATA_PATH.is_file() else DATA_URL
    data = pd.read_csv(source)

    missing_columns = sorted(set(required_columns) - set(data.columns))
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    return data


In [ ]:
# RUN THIS CELL FIRST
section_3_columns = ["risk_score_t", "gagne_sum_t", "race"]
df = load_data(section_3_columns)


## Variables used in this section

Each row represents one patient in one year.

- `risk_score_t`: the existing commercial score for year t. Higher values rank
  a row as higher risk under the current policy.
- `gagne_sum_t`: the number of active chronic conditions recorded in year t.
- `race`: the self-reported audit group, `black` or `white`.

You will create four additional columns or objects:

- `audit`: a smaller analysis table containing the three columns above;
- `risk_percentile`: a row's position in the pooled score distribution, from
  low to high, calculated before separating rows by race;
- `risk_bin`: one of 100 nearly equal-sized score ranges, numbered 1 to 100;
  and
- `commercial_selected`: a Boolean column that is `True` for exactly
  `number_selected = ceil(0.03 × n)` rows with the highest commercial scores.


## Task 1 — Create comparable commercial-risk ranges

### 1A — Create the analysis table

Create `audit` by selecting `section_3_columns` from `df` and making a copy.
Display its first five rows.


In [ ]:
# TODO


### 1B — Put scores on a pooled percentile scale

Add `risk_percentile` to `audit`. A pooled percentile ranks all patient rows together.
Rows tied on `risk_score_t` should receive the same average percentile.

**Hint:** `rank(method="average", pct=True)` returns values from 0 to 1. Multiply by 100 to express them as percentiles.

If you haven't used this method, read the [pandas documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rank.html) of `rank` to understand how it works. How can we use the `method` and `pct` arguments?


In [ ]:
# TODO


### 1C — Form 100 narrow score ranges

Create `stable_rank` by giving each row its position in the score distribution. The lowest score receives position 1 and the highest receives the number of rows. These positions stay aligned with the original rows. Then divide `stable_rank` into 100 nearly equal-sized groups and store the group number in `audit["risk_bin"]`.

**Example:**

Suppose that we only have 10 observations and the `risk_score_t` column is:
```
[22, 14, 4, 2, 3, 1, 35, 10, 7, 20]
```
then `stable_rank` is:
```
[9, 7, 4, 2, 3, 1, 10, 6, 5, 8]
```
if we divide this `stable_rank` into 5 nearly equal-sized groups, the group number is:
```
[5, 4, 2, 1, 2, 1, 5, 3, 3, 4]
```
we add this as the `risk_bin` column.

**Hints:**

- `rank(method="first")` gives tied scores different positions in their
  original row order.
- `pd.qcut(..., q=100, labels=False)` numbers groups from 0 to 99; add 1 so
  `risk_bin` runs from 1 to 100.
- If you haven't seen `qcut` method before, google/ask an LLM to learn how they work; or read the [pandas documentation](https://pandas.pydata.org/docs/reference/api/pandas.qcut.html) of `qcut`.

In [ ]:
# TODO


### 1D — Check the score ranges

Report the number of distinct `risk_bin` values and the smallest and largest
number of rows in a bin. You should find 100 bins containing either 487 or 488
rows each.


In [ ]:
# TODO


## Task 2 — Compare recorded active conditions within score ranges

### 2A — Build the score-range summary table

**Goal:** For every combination of score range (`risk_bin`) and race, compute
the average number of recorded active chronic conditions. Store the results in
a new DataFrame named `illness_by_risk`.

**What one row means.** Each row of `illness_by_risk` summarizes a *group* of
patients in `audit`, not a single patient-year. For example, the row
with `risk_bin == 50` and `race == "black"` summarizes all Black patients whose risk score falls in bin 50.

**Expected size.** There are 100 bins and 2 races, and every bin contains both
races, so `illness_by_risk` should have 100 × 2 = **200 rows** and these five
columns:

| Column | What it contains |
|---|---|
| `risk_bin` | The bin number, from 1 to 100 |
| `race` | `"black"` or `"white"`|
| `risk_percentile` | The **mean** of `risk_percentile` (the pooled percentile from Task 1) for the rows in this group |
| `mean_chronic_conditions` | The **mean** of `gagne_sum_t` for the rows in this group |
| `rows` | The **number** of patients in this group |


**Your task:**

Try to create the `illness_by_risk` DataFrame yourself.

**Hint:**

Feel free to use the code cell below as a template. Replace the three `...` placeholders in the code cell below to create the DataFrame `illness_by_risk`.


In [ ]:
summary_rows = []

for bin_number in range(1, 101):
    for race_name in ["black", "white"]:
        in_group = (
            (audit["risk_bin"] == bin_number)
            & (audit["race"] == race_name)
        )
        group_rows = audit.loc[in_group]

        summary_rows.append(
            {
                "risk_bin": bin_number,
                "race": race_name,
                "risk_percentile": ...,
                "mean_chronic_conditions": ...,
                "rows": ...,
            }
        )

illness_by_risk = pd.DataFrame(summary_rows)
illness_by_risk.head()


### 2B — Put the summary rows in plotting order

Sort `illness_by_risk` first by `race` and then by `risk_percentile`. Display
the first five rows.

In [ ]:
# TODO


### 2C — Plot the comparison

**Goal:** See whether Black and White patients with *similar* commercial risk
scores have *similar* numbers of recorded active chronic conditions. If the
score treated the two groups the same way, the two lines in the plot would lie
roughly on top of each other.

**What the plot shows.** The code cell below draws one line per race from
`illness_by_risk`:

- horizontal axis: `risk_percentile` (where the group sits in the pooled risk
  ranking);
- vertical axis: `mean_chronic_conditions` (how sick the group is, on
  average); and
- each point is one `risk_bin`–`race` summary row from 2A, not an individual patient. The 100 points for each race are connected from lower to higher risk.

**Your task:**

1. Make sure you have completed 2A (the five columns) and 2B (the sorting).
   The plot will be wrong or fail otherwise.
2. Run the code cell below. It is complete plotting code provided by the instructor, **you do not need to edit it**.
3. Use the figure to answer the **Read the figure** question that follows.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for race, style in RACE_STYLES.items():
    plot_data = illness_by_risk.loc[illness_by_risk["race"] == race]
    ax.plot(
        plot_data["risk_percentile"],
        plot_data["mean_chronic_conditions"],
        label=race.title(),
        linewidth=2,
        markersize=3,
        markevery=5,
        **style,
    )

ax.set(
    xlabel="Commercial-risk percentile (pooled)",
    ylabel="Mean recorded active chronic conditions",
    title="Recorded active chronic conditions across commercial-risk percentiles",
)
ax.legend(title="Audit group")
plt.show()


**Read the figure:**
Describe the pattern of the two lines. Which line tends to be higher at each commercial risk percentile? What does it mean?

**Your observation:** [Write a few sentences here.]


## Task 3 — Describe the fixed top-3% capacity group

The program chooses the patients with the highest top 3% commercial risk scores. We will examine some characteristics of the selected patients.

### 3A — Translate 3% capacity into a number of rows

Create two scalar variables:

- `selection_fraction`: the decimal value `0.03`, meaning 3% of all rows in
  the pooled `audit` DataFrame; and
- `number_selected`: the integer number of rows the program can accept. Compute
  it by multiplying `selection_fraction` by `len(audit)`, rounding upward, and
  converting the result to an integer.

Display `number_selected`. With 48,784 rows, it should equal 1,464. Later steps
will use this value to select exactly 1,464 rows.

**Note:** Make sure that `number_selected` has an `int` type.

In [ ]:
# TODO


### 3B — Rank rows for the commercial decision

**Goal:** Put all rows of `audit` in the order the program would admit them,
with the highest commercial risk score first.

**Your task:** Create a DataFrame named `commercial_ranking` that contains the
rows of `audit` sorted by `risk_score_t` from highest to lowest. Do not reset
the index: 3C uses the original row labels to find the selected rows in
`audit`.

Display the first five rows and check that they have the largest risk scores.


In [ ]:
# TODO


### 3C — Mark exactly the selected rows

**Goal:** Add a column `commercial_selected` to `audit` that is `True` for
the rows the program admits (the top `number_selected` rows in
`commercial_ranking`) and `False` for every other row.


**Hint:**
Feel free to use the code cell below as a template.

The last line counts the `True` values. It should equal `number_selected` (1,464).


In [ ]:
selected_index = commercial_ranking.head(...).index

audit["commercial_selected"] = False
audit.loc[..., "commercial_selected"] = True

audit["commercial_selected"].sum()


### 3D — Calculate one summary row for each race

**Goal:** For each race, describe who was admitted to the program: 
- how many were selected
- what share of the selected group they make up
- how likely a
patient of that race is to be selected
- how sick the selected patients are.

For each race, let's create a dictionary with the four values above. Use the code cell below as a template.

**Your task:** Replace the four `...` placeholders in the dictionary:

| Key | What to compute |
|---|---|
| `selected_rows` | The number of rows in `selected_race_rows` |
| `fraction_of_selected` | `selected_rows` divided by the total number of selected rows: this race's share of everyone selected |
| `within_race_selection_rate` | `selected_rows` divided by the total number of rows for this race: the share of this race's rows that were selected |
| `mean_chronic_conditions` | The mean of the number of chronic conditions among the selected patients in this race |

In [ ]:
selected = audit.loc[audit["commercial_selected"]].copy()
top_3_rows = []

for race_name in ["black", "white"]:
    race_rows = audit.loc[audit["race"] == race_name]
    selected_race_rows = selected.loc[selected["race"] == race_name]

    top_3_rows.append(
        {
            "race": race_name,
            "selected_rows": ...,
            "fraction_of_selected": ...,
            "within_race_selection_rate": ...,
            "mean_chronic_conditions": ...,
        }
    )


### 3E — Make a summary table

Run the following code cell as is to convert a list of dictionaries `top_3_rows` into a pandas DataFrame and display it.


In [ ]:
top_3_summary = pd.DataFrame(...).set_index(...)
top_3_summary.round(4)


**Read the table:**

- Among the selected patients, how many are Black and what share of the selected group do they make up? Compare the computed share with the share of the total population that is Black.
- Compare the mean number of chronic conditions among the selected patients in each race. What does this suggest about the racial bias in the program?